#https://www.dha.com.tr/1560000 2659181


In [60]:
import aiohttp
import asyncio
import csv
import os
from bs4 import BeautifulSoup
import re
import time

In [67]:

async def page_extractor(session, article_id):
    print(f"Processing article ID: {article_id}")
    url = f"https://www.dha.com.tr/{article_id}"
    try:
        async with session.get(url, timeout=10) as response:
            html = await response.text()
            soup = BeautifulSoup(html, 'html.parser')



        headline_tag = soup.find('h1')
        headline = headline_tag.get_text(strip=True) if headline_tag else None

        description_tag = soup.find('h2')
        description = description_tag.get_text(strip=True) if description_tag else None

        articleBody = "\n\n".join(
            p.get_text(strip=True)
            for p in soup.find_all('p')
            if not p.get_text(strip=True).startswith('Son Güncellenme')
        )

        articleBody2=articleBody.replace("UYARI: www.dha.com.tr internet sitesinde yayınlanan yazı, haber ve fotoğrafların her türlü telif hakkı Demirören Ajansı A.Ş.’ye aittir. İzin alınmadan, kaynak gösterilerek dahi kullanılamaz.", "No Body")


        meta_line_node = soup.find(string=re.compile(r'Son\s+Güncellenme'))
        meta_line = meta_line_node.strip() if meta_line_node else ''
        dates = re.findall(r'\d{2}\.\d{2}\.\d{4}', meta_line)
        dpublishdate = dates[0] if len(dates) >= 1 else None

        author_line = soup.find(string=re.compile(r'DHA\)'))
        if author_line:
            author_line = author_line.strip()
            author = author_line.split('|', 1)[1].split('/', 1)[0].strip() if '|' in author_line else None
        else:
            author = None

        if "DHA" in author:
            author = None


        return {
            "id": article_id,
            "headline": headline,
            "description": description,
            "articleBody": articleBody2,
            "dpublishdate": dpublishdate,
            "author": author
        }
    except Exception as e:
        print(f"Error processing article {article_id}: {e}")
        return None

In [72]:

async def main(start,end):
    csv_file = 'dha_articles.csv'
    fieldnames = ["id", "headline", "description", "articleBody","dpublishdate", "author"]
    write_header = not os.path.exists(csv_file) or os.path.getsize(csv_file) == 0

    start_time = time.time()
    async with aiohttp.ClientSession() as session:
        tasks = [page_extractor(session, i) for i in range(start, end)]
        results = await asyncio.gather(*tasks)
        results = [r for r in results if r is not None]
    with open(csv_file, 'a', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        for row in results:
            writer.writerow(row)
    end_time = time.time()
    print(f"Time taken to get the details of profiles: {end_time - start_time} seconds")



In [77]:
async def main1(start,end):
    while(start <end):
        await main(start,start+50)
        start += 50


In [78]:
await main1(1560000, 1561000)

Processing article ID: 1560000
Processing article ID: 1560001
Processing article ID: 1560002
Processing article ID: 1560003
Processing article ID: 1560004
Processing article ID: 1560005
Processing article ID: 1560006
Processing article ID: 1560007
Processing article ID: 1560008
Processing article ID: 1560009
Processing article ID: 1560010
Processing article ID: 1560011
Processing article ID: 1560012
Processing article ID: 1560013
Processing article ID: 1560014
Processing article ID: 1560015
Processing article ID: 1560016
Processing article ID: 1560017
Processing article ID: 1560018
Processing article ID: 1560019
Processing article ID: 1560020
Processing article ID: 1560021
Processing article ID: 1560022
Processing article ID: 1560023
Processing article ID: 1560024
Processing article ID: 1560025
Processing article ID: 1560026
Processing article ID: 1560027
Processing article ID: 1560028
Processing article ID: 1560029
Processing article ID: 1560030
Processing article ID: 1560031
Processi